In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import os

def main():
    video_path = 'files/21117-315137086.mp4' 
    output_dir = 'runs/detect'
    os.makedirs(output_dir, exist_ok=True)
    output_video_path = os.path.join(output_dir, 'retail_analytics_output.mp4')
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open input video {video_path}")
        return
        
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Loaded video: {width}x{height} @ {fps:.2f} FPS. Total frames: {total_frames}")
    
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    model = YOLO('yolo26n.pt')
    
    video_name = os.path.basename(video_path)
    
    if "21117" in video_name:
        print("Supermarket Checkout context detected. Mapping checkout lanes (2880x1620)...")
        ZONES = {
            "ZONE_LANE_1": {
                "label": "Checkout Lane 1 (Left)",
                "bbox": [50, 300, 750, 1500],
                "color": (255, 0, 255)
            },
            "ZONE_LANE_2": {
                "label": "Checkout Lane 2 (Mid-Left)",
                "bbox": [800, 300, 1450, 1500],
                "color": (0, 255, 255)
            },
            "ZONE_LANE_3": {
                "label": "Checkout Lane 3 (Mid-Right)",
                "bbox": [1500, 300, 2150, 1500],
                "color": (0, 165, 255)
            },
            "ZONE_LANE_4": {
                "label": "Checkout Lane 4 (Right)",
                "bbox": [2200, 300, 2800, 1500],
                "color": (255, 255, 0)
            }
        }
    else:
        print("Shopping Corridor context detected. Mapping store zones (1920x1080)...")
        ZONES = {
            "ZONE_PANDORA": {
                "label": "Pandora Storefront",
                "bbox": [1150, 300, 1600, 700],
                "color": (255, 0, 255)
            },
            "ZONE_EAT": {
                "label": "EAT. Seating Area",
                "bbox": [1000, 550, 1620, 850],
                "color": (0, 255, 255)
            },
            "ZONE_ESCALATOR": {
                "label": "Escalator Area",
                "bbox": [750, 300, 950, 850],
                "color": (0, 165, 255)
            },
            "ZONE_WALKWAY": {
                "label": "Walkway Corridor",
                "bbox": [100, 450, 650, 1000],
                "color": (255, 255, 0)
            }
        }
        
    scale_factor = width / 1920.0
    panel_w = int(420 * scale_factor)
    panel_h = int(290 * scale_factor)
    
    track_data = {}
    peak_occupancy = 0
    total_entries = 0
    
    heatmap_acc = np.zeros((height, width), dtype=np.float32)
    current_frame_idx = 0
    
    def is_point_in_box(px, py, box):
        x1, y1, x2, y2 = box
        return x1 <= px <= x2 and y1 <= py <= y2

    print("Processing video frames (Press 'q' in the window to quit early)...")
    
    window_name = "Retail Customer Intelligence Dashboard"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1280, 720)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        current_time_sec = current_frame_idx / fps
        
        results = model.track(frame, persist=True, tracker='bytetrack.yaml', classes=[0], verbose=False)
        
        current_frame_ids = []
        active_bbox_info = []
        
        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.int().cpu().tolist()
            
            for box, track_id in zip(boxes, track_ids):
                current_frame_ids.append(track_id)
                x1, y1, x2, y2 = map(int, box)
                
                cx = (x1 + x2) // 2
                cy = y2
                
                occupied_zone = None
                for zone_id, zone_info in ZONES.items():
                    if is_point_in_box(cx, cy, zone_info["bbox"]):
                        occupied_zone = zone_id
                        break
                        
                if track_id not in track_data:
                    total_entries += 1
                    track_data[track_id] = {
                        "entry_time": current_time_sec,
                        "last_seen": current_time_sec,
                        "active": True,
                        "zones_visited": set(),
                        "zone_frames": {z: 0 for z in ZONES},
                        "total_frames_active": 1,
                        "last_active_zone": occupied_zone
                    }
                else:
                    track_data[track_id]["last_seen"] = current_time_sec
                    track_data[track_id]["active"] = True
                    track_data[track_id]["total_frames_active"] += 1
                    
                if occupied_zone:
                    track_data[track_id]["zones_visited"].add(occupied_zone)
                    track_data[track_id]["zone_frames"][occupied_zone] += 1
                    track_data[track_id]["last_active_zone"] = occupied_zone
                    
                rad = int(20 * scale_factor)
                cv2.circle(heatmap_acc, (cx, cy), rad, 1, thickness=-1)
                
                dwell_sec = track_data[track_id]["total_frames_active"] / fps
                
                if dwell_sec < 2.0:
                    dwell_category = "Pass-through"
                    bbox_color = (0, 255, 0)
                elif dwell_sec < 5.0:
                    dwell_category = "Browsing"
                    bbox_color = (0, 255, 255)
                else:
                    dwell_category = "Engaged"
                    bbox_color = (0, 0, 255)
                    
                active_bbox_info.append((track_id, (x1, y1, x2, y2), (cx, cy), occupied_zone, dwell_category, bbox_color))
                
        for track_id in track_data:
            if track_id not in current_frame_ids:
                track_data[track_id]["active"] = False
                
        current_occupancy = len(current_frame_ids)
        if current_occupancy > peak_occupancy:
            peak_occupancy = current_occupancy
            
        heatmap_norm = cv2.normalize(heatmap_acc, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        heatmap_colored = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)
        
        annotated_frame = cv2.addWeighted(frame, 0.65, heatmap_colored, 0.35, 0)
        
        overlay = annotated_frame.copy()
        for zone_id, zone_info in ZONES.items():
            zx1, zy1, zx2, zy2 = zone_info["bbox"]
            color = zone_info["color"]
            label = zone_info["label"]
            
            zone_count = sum(1 for item in active_bbox_info if item[3] == zone_id)
            
            cv2.rectangle(overlay, (zx1, zy1), (zx2, zy2), color, thickness=-1)
            cv2.rectangle(annotated_frame, (zx1, zy1), (zx2, zy2), color, thickness=int(2 * scale_factor))
            
            label_text = f"{label} (Occupancy: {zone_count})"
            font_sz = 0.6 * scale_factor
            cv2.putText(annotated_frame, label_text, (zx1, zy1 - 10), cv2.FONT_HERSHEY_SIMPLEX, font_sz, color, int(2 * scale_factor), cv2.LINE_AA)
            
        annotated_frame = cv2.addWeighted(overlay, 0.15, annotated_frame, 0.85, 0)
        
        for track_id, (x1, y1, x2, y2), (cx, cy), occupied_zone, category, color in active_bbox_info:
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, int(2 * scale_factor))
            
            dwell_sec = track_data[track_id]["total_frames_active"] / fps
            
            box_label = f"ID: {track_id} | {dwell_sec:.1f}s | {category}"
            font_sz = 0.5 * scale_factor
            cv2.putText(annotated_frame, box_label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, font_sz, color, int(2 * scale_factor), cv2.LINE_AA)
            cv2.circle(annotated_frame, (cx, cy), int(5 * scale_factor), color, -1)
            
        completed_dwells = [d["total_frames_active"] / fps for d in track_data.values()]
        avg_dwell = np.mean(completed_dwells) if completed_dwells else 0.0
        
        zone_avg_dwells = {}
        for zone_id in ZONES:
            zone_active_dwells = []
            for d in track_data.values():
                if d["zone_frames"][zone_id] > 0:
                    zone_active_dwells.append(d["zone_frames"][zone_id] / fps)
            zone_avg_dwells[zone_id] = np.mean(zone_active_dwells) if zone_active_dwells else 0.0
            
        sorted_zones = sorted(zone_avg_dwells.items(), key=lambda x: x[1], reverse=True)
        
        if "21117" in video_name:
            counts = [sum(1 for item in active_bbox_info if item[3] == zid) for zid in ZONES]
            max_queue = max(counts) if counts else 0
            health_score = max(0.0, min(100.0, 100.0 - (max_queue * 15.0)))
        else:
            commercial_dwell = (zone_avg_dwells["ZONE_PANDORA"] + zone_avg_dwells["ZONE_EAT"]) / 2.0
            walkway_dwell = zone_avg_dwells["ZONE_WALKWAY"]
            health_score = 50.0
            if commercial_dwell > 0:
                health_score += min(30.0, commercial_dwell * 5)
            if walkway_dwell > 0:
                health_score -= min(15.0, (walkway_dwell - 1.0) * 3)
            health_score = max(0.0, min(100.0, health_score))
        
        cv2.rectangle(annotated_frame, (10, 10), (10 + panel_w, 10 + panel_h), (30, 30, 30), thickness=-1)
        cv2.rectangle(annotated_frame, (10, 10), (10 + panel_w, 10 + panel_h), (180, 180, 180), thickness=2)
        
        font_sz = 0.6 * scale_factor
        cv2.putText(annotated_frame, "STORE MANAGER BUSINESS PORTAL", (20, int(35 * scale_factor)), cv2.FONT_HERSHEY_SIMPLEX, font_sz, (0, 255, 255), 2, cv2.LINE_AA)
        cv2.line(annotated_frame, (20, int(45 * scale_factor)), (10 + panel_w - 10, int(45 * scale_factor)), (100, 100, 100), 1)
        
        font_sz = 0.5 * scale_factor
        cv2.putText(annotated_frame, f"Current Occupancy: {current_occupancy}", (20, int(70 * scale_factor)), cv2.FONT_HERSHEY_SIMPLEX, font_sz, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(annotated_frame, f"Peak Occupancy: {peak_occupancy}", (20, int(90 * scale_factor)), cv2.FONT_HERSHEY_SIMPLEX, font_sz, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(annotated_frame, f"Total Foot Traffic: {total_entries}", (20, int(110 * scale_factor)), cv2.FONT_HERSHEY_SIMPLEX, font_sz, (255, 255, 255), 1, cv2.LINE_AA)
        cv2.putText(annotated_frame, f"Avg Store Dwell: {avg_dwell:.1f}s", (20, int(130 * scale_factor)), cv2.FONT_HERSHEY_SIMPLEX, font_sz, (255, 255, 255), 1, cv2.LINE_AA)
        
        health_color = (0, 255, 0) if health_score >= 60 else ((0, 165, 255) if health_score >= 40 else (0, 0, 255))
        font_sz = 0.55 * scale_factor
        cv2.putText(annotated_frame, f"Store Health: {health_score:.1f}/100", (20, int(160 * scale_factor)), cv2.FONT_HERSHEY_SIMPLEX, font_sz, health_color, 2, cv2.LINE_AA)
        
        font_sz = 0.5 * scale_factor
        cv2.putText(annotated_frame, "ZONE ENGAGEMENT LEADERBOARD", (20, int(195 * scale_factor)), cv2.FONT_HERSHEY_SIMPLEX, font_sz, (0, 200, 255), 2, cv2.LINE_AA)
        cv2.line(annotated_frame, (20, int(202 * scale_factor)), (10 + panel_w - 10, int(202 * scale_factor)), (100, 100, 100), 1)
        
        y_offset = int(225 * scale_factor)
        font_sz = 0.45 * scale_factor
        for rank, (zone_id, val) in enumerate(sorted_zones[:3], start=1):
            zone_name = ZONES[zone_id]["label"]
            leaderboard_text = f"#{rank} {zone_name[:20]:<20} | Avg: {val:.1f}s"
            cv2.putText(annotated_frame, leaderboard_text, (20, y_offset), cv2.FONT_HERSHEY_SIMPLEX, font_sz, (220, 220, 220), 1, cv2.LINE_AA)
            y_offset += int(20 * scale_factor)
            
        progress = (current_frame_idx / total_frames) * 100
        progress_text = f"Frame: {current_frame_idx}/{total_frames} ({progress:.1f}%)"
        font_sz = 0.5 * scale_factor
        cv2.putText(annotated_frame, progress_text, (width - int(250 * scale_factor), height - int(20 * scale_factor)), cv2.FONT_HERSHEY_SIMPLEX, font_sz, (255, 255, 255), 1, cv2.LINE_AA)
        
        cv2.imshow(window_name, annotated_frame)
        
        out.write(annotated_frame)
        current_frame_idx += 1
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Execution stopped by user.")
            break
        
    cap.release()
    out.release()
    cv2.destroyAllWindows()
    
    print("\nProcessing complete!")
    print(f"Annotated output video saved successfully to: {output_video_path}")
    print("\n--- Final Store Telemetry ---")
    print(f"Total Customer Traffic: {total_entries}")
    print(f"Peak Customer Concurrency: {peak_occupancy}")
    
    completed_dwells = [d["total_frames_active"] / fps for d in track_data.values()]
    avg_dwell = np.mean(completed_dwells) if completed_dwells else 0.0
    print(f"Average Customer Visit Duration: {avg_dwell:.2f} seconds")
    
    print("\nZone Performance Summary:")
    for zone_id, zone_info in ZONES.items():
        label = zone_info["label"]
        visits = sum(1 for d in track_data.values() if d["zone_frames"][zone_id] > 0)
        zone_dwells = [d["zone_frames"][zone_id] / fps for d in track_data.values() if d["zone_frames"][zone_id] > 0]
        avg_zone_dwell = np.mean(zone_dwells) if zone_dwells else 0.0
        print(f"  - {label}: {visits} unique shoppers, Avg Dwell: {avg_zone_dwell:.2f} seconds")

if __name__ == "__main__":
    main()


Loaded video: 2880x1620 @ 25.00 FPS. Total frames: 150
Supermarket Checkout context detected. Mapping checkout lanes (2880x1620)...
Processing video frames (Press 'q' in the window to quit early)...
Execution stopped by user.

Processing complete!
Annotated output video saved successfully to: runs/detect\retail_analytics_output.mp4

--- Final Store Telemetry ---
Total Customer Traffic: 49
Peak Customer Concurrency: 15
Average Customer Visit Duration: 0.78 seconds

Zone Performance Summary:
  - Checkout Lane 1 (Left): 10 unique shoppers, Avg Dwell: 0.32 seconds
  - Checkout Lane 2 (Mid-Left): 13 unique shoppers, Avg Dwell: 0.95 seconds
  - Checkout Lane 3 (Mid-Right): 9 unique shoppers, Avg Dwell: 0.73 seconds
  - Checkout Lane 4 (Right): 12 unique shoppers, Avg Dwell: 0.63 seconds
